In [1]:
# --- Step 1: Stratified 80/20 split ---
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Load the cleaned data (the 74,549 x 9 file from the cleaning notebook)
df = pd.read_csv('data/cleaned_train.csv')

SEED = 42  # fixed seed: record this in your README, don't change it later

# Split the WHOLE dataframe (not just the text), so "Unnamed: 0" and the
# drug name stay attached to each row
train_df, val_df = train_test_split(
    df,                          # the data to split
    test_size=0.2,               # 20% goes to validation
    stratify=df['condition'],    # keep the same class mix in both parts
    random_state=SEED            # makes the split reproducible
)

# Reset the row numbers (the old index is kept in "Unnamed: 0" anyway)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Check 1: sizes
print('Train:', train_df.shape)
print('Validation:', val_df.shape)

# Check 2: class proportions side by side (should be almost identical)
check = pd.DataFrame({
    'train_share': train_df['condition'].value_counts(normalize=True),
    'val_share': val_df['condition'].value_counts(normalize=True),
    'val_rows': val_df['condition'].value_counts()
})
check['difference'] = (check['train_share'] - check['val_share']).abs()
check = check.sort_values('val_rows', ascending=False)
check

Train: (59639, 9)
Validation: (14910, 9)


,train_share,val_share,val_rows,difference
condition,,,,
Birth Control,0.242945,0.242991,3623,4.622829e-05
Depression,0.083486,0.083501,1245,1.536744e-05
Weight Management,0.067959,0.067941,1013,1.790676e-05
Pain,0.058435,0.058417,871,1.774706e-05
Anxiety,0.057261,0.057277,854,1.580716e-05
Acne,0.052885,0.052918,789,3.264781e-05
Bipolar Disorder,0.038297,0.038296,571,6.421376e-07
Insomnia,0.037509,0.037559,560,4.967289e-05
ADHD,0.029008,0.029041,433,3.304816e-05


In [3]:
# Ask the vectorizer how it splits a sentence into tokens
# analyzer = TfidfVectorizer().build_analyzer()

# Test string with contractions (my own example, not from your data)
# print(analyzer("I didn't sleep and I can't focus"))

# How many train reviews contain an apostrophe contraction?
# (str.contains with a regex; .mean() turns True/False into a share)
# train_df['review_clean'].str.contains("n't", case=False).mean()

In [4]:
# True/False per validation row: does this exact text also appear in train?
val_in_train = val_df['review_clean'].isin(train_df['review_clean'])

print('Validation reviews also in train:', val_in_train.sum())
print('Share of validation:', val_in_train.mean())

# Look at a few of them, with their conditions
val_df[val_in_train][['review_clean', 'condition']].head(10)

Validation reviews also in train: 3
Share of validation: 0.00020120724346076458


In [5]:
# For the overlapping texts, compare the condition in val with the condition(s) in train
overlap = val_df[val_in_train][['review_clean', 'condition']].merge(
    train_df[['review_clean', 'condition']],
    on='review_clean',
    suffixes=('_val', '_train')
)
overlap['same_condition'] = overlap['condition_val'] == overlap['condition_train']
overlap['same_condition'].value_counts()

same_condition
True    3
Name: count, dtype: int64

In [6]:
# Keyword to search for each class (my picks, change as you like)
keywords = {
    'Acne': 'acne',
    'Insomnia': 'insomnia',
    'Migraine': 'migraine',
    'Panic Disorder': 'panic',
    'Erectile Dysfunction': 'erectile'
}

rows = []
for condition, word in keywords.items():
    subset = train_df[train_df['condition'] == condition]            # reviews of this class
    share = subset['review_clean'].str.contains(word, case=False).mean()  # share mentioning the word
    rows.append({'condition': condition, 'keyword': word, 'share_mentioning': share})

pd.DataFrame(rows)

,condition,keyword,share_mentioning
0,Acne,acne,0.770450
1,Insomnia,insomnia,0.297273
2,Migraine,migraine,0.797024
3,Panic Disorder,panic,0.736579
4,Erectile Dysfunction,erectile,0.122124


In [7]:
# Most common first words and last words
first_words = train_df['review_clean'].str.split().str[0].value_counts().head(20)
last_words = train_df['review_clean'].str.split().str[-1].value_counts().head(20)

pd.DataFrame({'first_word': first_words.index, 'first_count': first_words.values,
              'last_word': last_words.index, 'last_count': last_words.values})

,first_word,first_count,last_word,last_count
0,"""I",26441,"""",2932
1,"""I've",4362,"it.""",2088
2,"""This",3176,"me.""",1390
3,"""I'm",1872,"effects.""",696
4,"""My",1840,"again.""",631
5,"""After",874,"it!""",484
6,"""It",852,"life.""",427
7,"""The",838,"pill.""",402
8,"""So",633,"pain.""",398
9,"""Started",523,"medication.""",389


In [8]:
# Keep only validation rows that are NOT also in train
# (~ flips True/False, so this keeps the rows where val_in_train is False)
val_df = val_df[~val_in_train]

# Renumber the rows 0, 1, 2, ... (the old index is still kept in "Unnamed: 0")
val_df = val_df.reset_index(drop=True)

# Check 1: the size should now be 14,907 rows (14,910 minus 3)
print('Validation:', val_df.shape)

# Check 2: re-run the overlap test on the NEW val_df; the sum should be 0
print('Still in train:', val_df['review_clean'].isin(train_df['review_clean']).sum())

Validation: (14907, 9)
Still in train: 0


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Baseline settings (the decisions locked above)
vectorizer = TfidfVectorizer(lowercase=True, stop_words=None)

# Labels: the condition column of each split
y_train = train_df['condition']
y_val = val_df['condition']

# FIT on train only: learns the vocabulary and the IDF values, then converts train text
X_train = vectorizer.fit_transform(train_df['review_clean'])

# TRANSFORM only on validation: uses what train taught it, learns nothing new
X_val = vectorizer.transform(val_df['review_clean'])

# Check 1: shapes (rows should be 59,639 and 14,907)
print('X_train:', X_train.shape)
print('X_val:', X_val.shape)

# Check 2: both must have the SAME number of columns (the train vocabulary size)
print('Vocabulary size:', len(vectorizer.vocabulary_))

X_train: (59639, 34186)
X_val: (14907, 34186)
Vocabulary size: 34186


In [10]:
# The first 10 "column names" of X_train (the words), in column order
print(vectorizer.get_feature_names_out()[:10])

# Column number 5000 of X_train corresponds to this word
vectorizer.get_feature_names_out()[5000]

['00' '000' '00001' '000mg' '001' '0010782485900484' '0015' '002' '0020'
 '004']


'becauses'

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Baseline model: no class weighting, more iterations so it can converge
model = LogisticRegression(max_iter=1000)

# Learn from the train matrix and the train labels
model.fit(X_train, y_train)

# Predict the conditions of the validation reviews
y_pred = model.predict(X_val)

# Headline metric: macro F1 (every class counts equally)
print('Macro F1:', f1_score(y_val, y_pred, average='macro'))

Macro F1: 0.6961425398034867


In [12]:
from sklearn.metrics import classification_report

# Per-class precision, recall, F1 and support, as a dataframe
report = pd.DataFrame(
    classification_report(y_val, y_pred, output_dict=True)
).T

# Keep only the 30 class rows, weakest F1 first
report_classes = report.drop(['accuracy', 'macro avg', 'weighted avg'])
report_classes.sort_values('f1-score')

,precision,recall,f1-score,support
Major Depressive Disorder,0.538462,0.066038,0.117647,212.0
Generalized Anxiety Disorder,0.500000,0.079096,0.136585,177.0
Chronic Pain,0.569231,0.179612,0.273063,206.0
Anxiety and Stress,0.615385,0.228571,0.333333,245.0
Panic Disorder,0.516393,0.315000,0.391304,200.0
Abnormal Uterine Bleeding,0.733945,0.299625,0.425532,267.0
Muscle Spasm,0.851064,0.446927,0.586081,179.0
Anxiety,0.551357,0.667057,0.603714,853.0
Depression,0.537365,0.756627,0.628419,1245.0
Fibromyalgia,0.814208,0.598394,0.689815,249.0


In [13]:
# Confusion matrix: each ROW is a true condition, and shares sum to 1 across the row
cm = pd.crosstab(
    y_val, y_pred,
    normalize='index',          # divide by the row total (the true class size)
    rownames=['true'],          # replaces the unnamed "condition" label
    colnames=['predicted']      # replaces the "col_0" label you saw
)

# Sanity check 1: every row should sum to 1
print(cm.sum(axis=1).round(3).unique())

# Sanity check 2: the diagonal is each class's recall, so it should match your report
cm.loc['Birth Control', 'Birth Control']   # expect about 0.965

[1.]


np.float64(0.9654982059067071)

In [14]:
# Classes to inspect (edit this list to look at others)
weak = ['Major Depressive Disorder', 'Anxiety and Stress', 'Chronic Pain']

rows = []
for cls in weak:
    top3 = cm.loc[cls].sort_values(ascending=False).head(3)   # 3 most common predictions
    rows.append({
        'true_class': cls,
        'pred_1': f"{top3.index[0]} ({top3.iloc[0]:.2f})",
        'pred_2': f"{top3.index[1]} ({top3.iloc[1]:.2f})",
        'pred_3': f"{top3.index[2]} ({top3.iloc[2]:.2f})",
    })

pd.DataFrame(rows)

,true_class,pred_1,pred_2,pred_3
0,Major Depressive Disorder,Depression (0.70),Major Depressive Disorder (0.07),Anxiety (0.07)
1,Anxiety and Stress,Depression (0.43),Anxiety and Stress (0.23),Anxiety (0.21)
2,Chronic Pain,Pain (0.69),Chronic Pain (0.18),Fibromyalgia (0.02)


In [15]:
# Classes to inspect: the three weak ones we haven't seen yet, plus the
# two big siblings, so we can see the reverse direction (Anxiety, Depression)
to_check = ['Generalized Anxiety Disorder', 'Panic Disorder',
            'Abnormal Uterine Bleeding', 'Muscle Spasm',
            'Anxiety', 'Depression']

rows = []
for cls in to_check:
    top3 = cm.loc[cls].sort_values(ascending=False).head(3)   # 3 most common predictions
    rows.append({
        'true_class': cls,
        'pred_1': f"{top3.index[0]} ({top3.iloc[0]:.2f})",
        'pred_2': f"{top3.index[1]} ({top3.iloc[1]:.2f})",
        'pred_3': f"{top3.index[2]} ({top3.iloc[2]:.2f})",
    })

pd.DataFrame(rows)

,true_class,pred_1,pred_2,pred_3
0,Generalized Anxiety Disorder,Anxiety (0.43),Depression (0.34),Generalized Anxiety Disorder (0.08)
1,Panic Disorder,Anxiety (0.45),Panic Disorder (0.32),Depression (0.16)
2,Abnormal Uterine Bleeding,Birth Control (0.68),Abnormal Uterine Bleeding (0.30),Pain (0.01)
3,Muscle Spasm,Muscle Spasm (0.45),Pain (0.32),Insomnia (0.08)
4,Anxiety,Anxiety (0.67),Depression (0.17),Panic Disorder (0.03)
5,Depression,Depression (0.76),Anxiety (0.07),Insomnia (0.03)


In [16]:
# Mapping: small class -> larger sibling (the larger class's name is kept)
merge_map = {
    'Major Depressive Disorder': 'Depression',
    'Chronic Pain': 'Pain'
}

# New column built from the ORIGINAL 'condition' (which stays untouched).
# Safe to re-run, because it always starts from 'condition'.
train_df['condition_merged'] = train_df['condition'].replace(merge_map)
val_df['condition_merged'] = val_df['condition'].replace(merge_map)

# Check 1: rows unchanged (59,639 and 14,907) and 28 classes
print('Train:', train_df.shape, '| Val:', val_df.shape)
print('Classes in train:', train_df['condition_merged'].nunique())

# Check 2: duplicates WITHIN train (same text and same merged label)
dup_train = train_df.duplicated(subset=['review_clean', 'condition_merged'], keep=False)
print('Train rows involved in a duplicate:', dup_train.sum())

# Check 3: validation reviews whose text AND merged label also appear in train
val_key = val_df['review_clean'] + '||' + val_df['condition_merged']
train_key = train_df['review_clean'] + '||' + train_df['condition_merged']
print('Val rows also in train:', val_key.isin(train_key).sum())

# Check 4: the same text under two DIFFERENT merged labels (label conflicts)
conflicts = train_df.groupby('review_clean')['condition_merged'].nunique()
print('Texts in train under 2+ merged labels:', (conflicts > 1).sum())

Train: (59639, 10) | Val: (14907, 10)
Classes in train: 28
Train rows involved in a duplicate: 10
Val rows also in train: 0
Texts in train under 2+ merged labels: 0


In [17]:
# Same duplicate count, but using the ORIGINAL label instead of the merged one
dup_before = train_df.duplicated(subset=['review_clean', 'condition'], keep=False)
print('Duplicate rows before merge:', dup_before.sum())
print('Duplicate rows after merge:', dup_train.sum())

# Look at the duplicate rows side by side (add your date and drug columns if you like)
train_df[dup_train].sort_values('review_clean')[['review_clean', 'condition', 'condition_merged']]

Duplicate rows before merge: 10
Duplicate rows after merge: 10


In [18]:
# --- Remove the 5 duplicate copies from train, then rebuild everything ---

# Keep the first copy of each (text + merged label) repeat, drop the others
train_df = train_df.drop_duplicates(subset=['review_clean', 'condition_merged'], keep='first')

# Renumber the rows (the old index is still kept in "Unnamed: 0")
train_df = train_df.reset_index(drop=True)

# Check 1: expect 59,634 rows (59,639 minus 5)
print('Train:', train_df.shape)

# Check 2: no duplicates left; expect 0
print('Duplicates left:', train_df.duplicated(subset=['review_clean', 'condition_merged']).sum())

# Rebuild the vectorizer: same baseline settings, fitted on the NEW train only
vectorizer = TfidfVectorizer(lowercase=True, stop_words=None)
X_train = vectorizer.fit_transform(train_df['review_clean'])
X_val = vectorizer.transform(val_df['review_clean'])

# Rebuild the labels from the MERGED column
y_train = train_df['condition_merged']
y_val = val_df['condition_merged']

# Check 3: rows match, columns match, 28 classes
print('X_train:', X_train.shape, '| X_val:', X_val.shape)
print('Classes:', y_train.nunique(), y_val.nunique())

Train: (59634, 10)
Duplicates left: 0
X_train: (59634, 34186) | X_val: (14907, 34186)
Classes: 28 28


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

model = LogisticRegression(max_iter=1000)   # same baseline settings as before
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print('Macro F1 (28 classes):', f1_score(y_val, y_pred, average='macro'))

Macro F1 (28 classes): 0.7370933963388502


In [20]:
# Same model, but small classes count for more during training
model_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
model_bal.fit(X_train, y_train)
y_pred_bal = model_bal.predict(X_val)

print('Macro F1 (balanced):', f1_score(y_val, y_pred_bal, average='macro'))

Macro F1 (balanced): 0.7266413137580268


In [21]:
from sklearn.metrics import classification_report

# Per-class report for each model, as dataframes (drop the summary rows)
rep_plain = pd.DataFrame(classification_report(y_val, y_pred, output_dict=True)).T
rep_bal = pd.DataFrame(classification_report(y_val, y_pred_bal, output_dict=True)).T
summary_rows = ['accuracy', 'macro avg', 'weighted avg']
rep_plain = rep_plain.drop(summary_rows)
rep_bal = rep_bal.drop(summary_rows)

# Side-by-side comparison, one row per class
compare = pd.DataFrame({
    'support': rep_plain['support'],
    'prec_plain': rep_plain['precision'],
    'prec_bal': rep_bal['precision'],
    'rec_plain': rep_plain['recall'],
    'rec_bal': rep_bal['recall'],
    'f1_plain': rep_plain['f1-score'],
    'f1_bal': rep_bal['f1-score'],
})
compare['f1_change'] = compare['f1_bal'] - compare['f1_plain']   # positive = balanced is better

# Biggest losers first, so the classes balanced hurts are at the top
compare.sort_values('f1_change').round(3)

,support,prec_plain,prec_bal,rec_plain,rec_bal,f1_plain,f1_bal,f1_change
Erectile Dysfunction,141.0,0.966,0.703,0.809,0.922,0.880,0.798,-0.083
Bacterial Infection,157.0,0.814,0.591,0.726,0.828,0.768,0.690,-0.078
Urinary Tract Infection,189.0,0.865,0.683,0.746,0.799,0.801,0.737,-0.065
Anxiety,853.0,0.577,0.667,0.658,0.477,0.614,0.556,-0.058
Birth Control,3623.0,0.893,0.957,0.965,0.799,0.927,0.871,-0.056
Depression,1457.0,0.612,0.769,0.800,0.556,0.694,0.645,-0.049
High Blood Pressure,363.0,0.811,0.698,0.793,0.848,0.802,0.766,-0.036
Constipation,223.0,0.860,0.755,0.857,0.910,0.858,0.825,-0.033
Acne,789.0,0.918,0.829,0.876,0.907,0.896,0.866,-0.030
Osteoarthritis,163.0,0.857,0.590,0.589,0.785,0.698,0.674,-0.024


In [22]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Check the exact name of the drug column first
print(train_df.columns.tolist())
DRUG_COL = 'drugName'   # change this if your column is named differently

# One-hot encoder: one 0/1 column per drug; unseen drugs in validation become all zeros
encoder = OneHotEncoder(handle_unknown='ignore')

# FIT on train only (learns the list of drugs), then convert train
drug_train = encoder.fit_transform(train_df[[DRUG_COL]])   # double brackets: needs a 2D input

# TRANSFORM only on validation (no learning)
drug_val = encoder.transform(val_df[[DRUG_COL]])

# Check 1: rows match the dataframes, columns = number of train drugs
print('drug_train:', drug_train.shape, '| drug_val:', drug_val.shape)

# Check 2: validation rows whose drug never appears in train
unseen = ~val_df[DRUG_COL].isin(train_df[DRUG_COL])
print('Validation rows with an unseen drug:', unseen.sum())

# Same model settings as the text-only baseline (no class weighting)
model_drug = LogisticRegression(max_iter=1000)
model_drug.fit(drug_train, y_train)          # y_train = the 28-class merged labels
y_pred_drug = model_drug.predict(drug_val)

# Headline metric, for comparison with 0.737 (text-only, 28 classes)
print('Macro F1 (drug only):', f1_score(y_val, y_pred_drug, average='macro'))

['Unnamed: 0', 'uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_clean', 'condition_merged']
drug_train: (59634, 1160) | drug_val: (14907, 1160)
Validation rows with an unseen drug: 53
Macro F1 (drug only): 0.7411210757035488


In [23]:
from sklearn.metrics import classification_report

# The watch-list: classes whose reviews the text-only model sent elsewhere, plus their siblings
watch = ['Generalized Anxiety Disorder', 'Panic Disorder', 'Anxiety and Stress',
         'Anxiety', 'Depression',
         'Abnormal Uterine Bleeding', 'Birth Control',
         'Muscle Spasm', 'Pain',
         'Migraine', 'Migraine Prevention']

# Per-class reports (zero_division=0 avoids warnings if a class is never predicted)
summary_rows = ['accuracy', 'macro avg', 'weighted avg']
rep_text = pd.DataFrame(classification_report(y_val, y_pred, output_dict=True, zero_division=0)).T.drop(summary_rows)
rep_drug = pd.DataFrame(classification_report(y_val, y_pred_drug, output_dict=True, zero_division=0)).T.drop(summary_rows)

# Side by side: text-only vs. drug-only
compare_drug = pd.DataFrame({
    'support': rep_text['support'],
    'rec_text': rep_text['recall'],
    'rec_drug': rep_drug['recall'],
    'prec_text': rep_text['precision'],
    'prec_drug': rep_drug['precision'],
    'f1_text': rep_text['f1-score'],
    'f1_drug': rep_drug['f1-score'],
})
compare_drug['f1_change'] = compare_drug['f1_drug'] - compare_drug['f1_text']   # positive = drug-only is better

# Watch-list rows only
compare_drug.loc[watch].round(3)

,support,rec_text,rec_drug,prec_text,prec_drug,f1_text,f1_drug,f1_change
Generalized Anxiety Disorder,177.0,0.068,0.000,0.480,0.000,0.119,0.000,-0.119
Panic Disorder,200.0,0.315,0.000,0.521,0.000,0.393,0.000,-0.393
Anxiety and Stress,245.0,0.224,0.384,0.640,0.445,0.332,0.412,0.080
Anxiety,853.0,0.658,0.777,0.577,0.564,0.614,0.654,0.039
Depression,1457.0,0.800,0.747,0.612,0.654,0.694,0.698,0.004
Abnormal Uterine Bleeding,267.0,0.311,0.015,0.735,0.800,0.437,0.029,-0.407
Birth Control,3623.0,0.965,1.000,0.893,0.842,0.927,0.914,-0.013
Muscle Spasm,179.0,0.430,0.899,0.846,0.910,0.570,0.904,0.334
Pain,1077.0,0.892,0.899,0.717,0.896,0.795,0.898,0.102
Migraine,235.0,0.728,0.881,0.830,0.990,0.776,0.932,0.157


In [24]:
# Where do the drug-only model's predictions go for each watch-list class?
cm_drug = pd.crosstab(y_val, y_pred_drug, normalize='index',
                      rownames=['true'], colnames=['predicted'])

rows = []
for cls in watch:
    top3 = cm_drug.loc[cls].sort_values(ascending=False).head(3)   # 3 most common predictions
    rows.append({
        'true_class': cls,
        'pred_1': f"{top3.index[0]} ({top3.iloc[0]:.2f})",
        'pred_2': f"{top3.index[1]} ({top3.iloc[1]:.2f})",
        'pred_3': f"{top3.index[2]} ({top3.iloc[2]:.2f})",
    })

pd.DataFrame(rows)

,true_class,pred_1,pred_2,pred_3
0,Generalized Anxiety Disorder,Anxiety (0.42),Depression (0.24),Fibromyalgia (0.18)
1,Panic Disorder,Anxiety (0.59),Depression (0.36),Anxiety and Stress (0.04)
2,Anxiety and Stress,Depression (0.52),Anxiety and Stress (0.38),Migraine Prevention (0.05)
3,Anxiety,Anxiety (0.78),Depression (0.14),Migraine Prevention (0.03)
4,Depression,Depression (0.75),Anxiety (0.10),Anxiety and Stress (0.07)
5,Abnormal Uterine Bleeding,Birth Control (0.97),Weight Management (0.01),Abnormal Uterine Bleeding (0.01)
6,Birth Control,Birth Control (1.00),Abnormal Uterine Bleeding (0.00),ADHD (0.00)
7,Muscle Spasm,Muscle Spasm (0.90),Anxiety (0.08),Birth Control (0.02)
8,Pain,Pain (0.90),Migraine Prevention (0.03),Anxiety (0.02)
9,Migraine,Migraine (0.88),Anxiety (0.06),Pain (0.03)


In [25]:
# Same drug-only model, but on the ORIGINAL 30-class labels
y_train_30 = train_df['condition']
y_val_30 = val_df['condition']

model_drug_30 = LogisticRegression(max_iter=1000)
model_drug_30.fit(drug_train, y_train_30)            # drug_train is unchanged
y_pred_drug_30 = model_drug_30.predict(drug_val)

print('Macro F1 (drug only, 30 classes):', f1_score(y_val_30, y_pred_drug_30, average='macro'))

Macro F1 (drug only, 30 classes): 0.7025230670224123


In [26]:
# Text-only reference on the ORIGINAL 30-class labels (same settings as before)
model_text_30 = LogisticRegression(max_iter=1000)
model_text_30.fit(X_train, y_train_30)
y_pred_text_30 = model_text_30.predict(X_val)
print('Macro F1 (text only, 30 classes):', f1_score(y_val_30, y_pred_text_30, average='macro'))

# Per-class reports for both 30-class models
summary_rows = ['accuracy', 'macro avg', 'weighted avg']
rep_text_30 = pd.DataFrame(classification_report(y_val_30, y_pred_text_30, output_dict=True, zero_division=0)).T.drop(summary_rows)
rep_drug_30 = pd.DataFrame(classification_report(y_val_30, y_pred_drug_30, output_dict=True, zero_division=0)).T.drop(summary_rows)

compare_30 = pd.DataFrame({
    'support': rep_text_30['support'],
    'rec_text': rep_text_30['recall'],
    'rec_drug': rep_drug_30['recall'],
    'f1_text': rep_text_30['f1-score'],
    'f1_drug': rep_drug_30['f1-score'],
})
compare_30['f1_change'] = compare_30['f1_drug'] - compare_30['f1_text']   # positive = drug-only is better

# The two merged pairs first
pairs = ['Depression', 'Major Depressive Disorder', 'Pain', 'Chronic Pain']
compare_30.loc[pairs].round(3)

Macro F1 (text only, 30 classes): 0.6965985579640811


,support,rec_text,rec_drug,f1_text,f1_drug,f1_change
Depression,1245.0,0.756,0.697,0.628,0.620,-0.008
Major Depressive Disorder,212.0,0.066,0.094,0.118,0.165,0.047
Pain,871.0,0.837,0.871,0.709,0.805,0.097
Chronic Pain,206.0,0.175,0.209,0.270,0.320,0.050


In [27]:
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Put the text columns and the drug columns side by side (same rows, same order)
X_train_both = hstack([X_train, drug_train]).tocsr()
X_val_both = hstack([X_val, drug_val]).tocsr()

# Check 1: shapes. Expect (59634, 35346) and (14907, 35346): 34,186 + 1,160 columns
print('X_train_both:', X_train_both.shape)
print('X_val_both:', X_val_both.shape)

# Same settings as the other baselines (no class weighting); 28-class merged labels
model_both = LogisticRegression(max_iter=1000)
model_both.fit(X_train_both, y_train)
y_pred_both = model_both.predict(X_val_both)

# Headline metric: compare with 0.737 (text only) and 0.741 (drug only)
print('Macro F1 (text + drug):', f1_score(y_val, y_pred_both, average='macro'))

X_train_both: (59634, 35346)
X_val_both: (14907, 35346)
Macro F1 (text + drug): 0.8539661004802237


In [28]:
from sklearn.metrics import classification_report

# Per-class reports for the three 28-class models (drop the summary rows)
summary_rows = ['accuracy', 'macro avg', 'weighted avg']
rep_t = pd.DataFrame(classification_report(y_val, y_pred, output_dict=True, zero_division=0)).T.drop(summary_rows)
rep_d = pd.DataFrame(classification_report(y_val, y_pred_drug, output_dict=True, zero_division=0)).T.drop(summary_rows)
rep_b = pd.DataFrame(classification_report(y_val, y_pred_both, output_dict=True, zero_division=0)).T.drop(summary_rows)

# Side by side: text-only, drug-only, text + drug
compare_all = pd.DataFrame({
    'support': rep_t['support'],
    'rec_text': rep_t['recall'],  'rec_drug': rep_d['recall'],  'rec_both': rep_b['recall'],
    'prec_text': rep_t['precision'], 'prec_drug': rep_d['precision'], 'prec_both': rep_b['precision'],
    'f1_text': rep_t['f1-score'], 'f1_drug': rep_d['f1-score'], 'f1_both': rep_b['f1-score'],
})
compare_all['f1_vs_text'] = compare_all['f1_both'] - compare_all['f1_text']   # positive = combined beats text-only
compare_all['f1_vs_drug'] = compare_all['f1_both'] - compare_all['f1_drug']   # positive = combined beats drug-only

# Watch-list rows (the 'watch' list from the earlier cell)
compare_all.loc[watch].round(3)

,support,rec_text,rec_drug,rec_both,prec_text,prec_drug,prec_both,f1_text,f1_drug,f1_both,f1_vs_text,f1_vs_drug
Generalized Anxiety Disorder,177.0,0.068,0.000,0.266,0.480,0.000,0.653,0.119,0.000,0.378,0.259,0.378
Panic Disorder,200.0,0.315,0.000,0.390,0.521,0.000,0.634,0.393,0.000,0.483,0.090,0.483
Anxiety and Stress,245.0,0.224,0.384,0.494,0.640,0.445,0.602,0.332,0.412,0.543,0.210,0.130
Anxiety,853.0,0.658,0.777,0.778,0.577,0.564,0.714,0.614,0.654,0.745,0.130,0.091
Depression,1457.0,0.800,0.747,0.850,0.612,0.654,0.758,0.694,0.698,0.801,0.107,0.103
Abnormal Uterine Bleeding,267.0,0.311,0.015,0.363,0.735,0.800,0.735,0.437,0.029,0.486,0.049,0.457
Birth Control,3623.0,0.965,1.000,0.984,0.893,0.842,0.933,0.927,0.914,0.958,0.031,0.044
Muscle Spasm,179.0,0.430,0.899,0.944,0.846,0.910,0.929,0.570,0.904,0.936,0.366,0.032
Pain,1077.0,0.892,0.899,0.957,0.717,0.896,0.891,0.795,0.898,0.923,0.128,0.025
Migraine,235.0,0.728,0.881,0.932,0.830,0.990,0.969,0.776,0.932,0.950,0.175,0.018


In [29]:
# All 28 classes, biggest losers vs. text-only first
compare_all.sort_values('f1_vs_text').round(3)

,support,rec_text,rec_drug,rec_both,prec_text,prec_drug,prec_both,f1_text,f1_drug,f1_both,f1_vs_text,f1_vs_drug
Birth Control,3623.0,0.965,1.000,0.984,0.893,0.842,0.933,0.927,0.914,0.958,0.031,0.044
Smoking Cessation,243.0,0.926,0.893,0.984,0.983,1.000,0.996,0.953,0.943,0.990,0.036,0.046
Vaginal Yeast Infection,402.0,0.953,0.988,0.990,0.960,1.000,1.000,0.956,0.994,0.995,0.039,0.001
Acne,789.0,0.876,0.760,0.906,0.918,0.974,0.966,0.896,0.854,0.935,0.039,0.081
Bowel Preparation,234.0,0.885,0.906,0.923,0.958,1.000,1.000,0.920,0.951,0.960,0.040,0.009
Emergency Contraception,315.0,0.895,0.470,0.987,0.989,1.000,0.990,0.940,0.639,0.989,0.049,0.350
Abnormal Uterine Bleeding,267.0,0.311,0.015,0.363,0.735,0.800,0.735,0.437,0.029,0.486,0.049,0.457
Fibromyalgia,249.0,0.574,0.466,0.651,0.836,0.699,0.848,0.681,0.559,0.736,0.055,0.177
Weight Management,1013.0,0.917,0.962,0.978,0.870,0.963,0.987,0.893,0.962,0.983,0.090,0.020
Panic Disorder,200.0,0.315,0.000,0.390,0.521,0.000,0.634,0.393,0.000,0.483,0.090,0.483


In [30]:
# Which of the variables we need still exist in this notebook session?
names = ['merge_map', 'train_df', 'val_df', 'X_train', 'X_val', 'drug_train', 'drug_val',
         'y_train', 'y_val', 'vectorizer', 'encoder', 'model', 'model_drug', 'model_both',
         'y_pred', 'y_pred_drug', 'y_pred_both']
for n in names:
    print(n, '-> exists' if n in globals() else '-> MISSING')

# Shapes for the ones that exist (expect train_df 59,634 rows and val_df 14,907 rows)
if 'train_df' in globals(): print('train_df:', train_df.shape)
if 'val_df' in globals(): print('val_df:', val_df.shape)

merge_map -> exists
train_df -> exists
val_df -> exists
X_train -> exists
X_val -> exists
drug_train -> exists
drug_val -> exists
y_train -> exists
y_val -> exists
vectorizer -> exists
encoder -> exists
model -> exists
model_drug -> exists
model_both -> exists
y_pred -> exists
y_pred_drug -> exists
y_pred_both -> exists
train_df: (59634, 10)
val_df: (14907, 10)


In [31]:
import os, joblib
from sklearn.metrics import f1_score

# Safety check 1: rows and labels line up
assert train_df.shape[0] == X_train.shape[0] == drug_train.shape[0] == len(y_train) == 59634
assert val_df.shape[0] == X_val.shape[0] == drug_val.shape[0] == len(y_val) == len(y_pred_both) == 14907
assert y_train.nunique() == 28

# Safety check 2: the three predictions are the ones we expect (0.737, 0.741, 0.854)
for name, preds, expected in [('text-only', y_pred, 0.737),
                              ('drug-only', y_pred_drug, 0.741),
                              ('text+drug', y_pred_both, 0.854)]:
    score = f1_score(y_val, preds, average='macro')
    print(name, round(score, 3))
    assert abs(score - expected) < 0.002, f'{name} score is not what we expected'

# Everything the final steps need, saved as a set
os.makedirs('data/checkpoints', exist_ok=True)
to_save = {
    'train_df': train_df, 'val_df': val_df,
    'X_train': X_train, 'X_val': X_val,
    'drug_train': drug_train, 'drug_val': drug_val,
    'y_train': y_train, 'y_val': y_val, 'merge_map': merge_map,
    'vectorizer': vectorizer, 'encoder': encoder,
    'model': model, 'model_drug': model_drug, 'model_both': model_both,
    'y_pred': y_pred, 'y_pred_drug': y_pred_drug, 'y_pred_both': y_pred_both,
}
for name, obj in to_save.items():
    joblib.dump(obj, f'data/checkpoints/{name}.joblib')
print('Saved', len(to_save), 'objects')

text-only 0.737
drug-only 0.741
text+drug 0.854
Saved 17 objects


In [1]:
import joblib
import pandas as pd
from scipy.sparse import hstack
from sklearn.metrics import accuracy_score, f1_score

names = ['train_df', 'val_df', 'X_train', 'X_val', 'drug_train', 'drug_val',
         'y_train', 'y_val', 'merge_map', 'vectorizer', 'encoder',
         'model', 'model_drug', 'model_both', 'y_pred', 'y_pred_drug', 'y_pred_both']
for n in names:
    globals()[n] = joblib.load(f'data/checkpoints/{n}.joblib')

DRUG_COL = 'drugName'   # not saved in the checkpoints, so it's defined here

# Check: rows line up, and the score is the one we saved (about 0.854)
print(train_df.shape, X_train.shape, val_df.shape, X_val.shape)
print(round(f1_score(y_val, y_pred_both, average='macro'), 3))

(59634, 10) (59634, 34186) (14907, 10) (14907, 34186)
0.854


In [2]:
# Where does the text + drug model send the four anxiety classes' reviews? (28-class, validation)
anxiety_four = ['Anxiety', 'Anxiety and Stress', 'Generalized Anxiety Disorder', 'Panic Disorder']

# Row-normalised confusion matrix: each row sums to 1
cm_both = pd.crosstab(y_val, y_pred_both, normalize='index',
                      rownames=['true'], colnames=['predicted'])

# Sanity check: the diagonal is recall, so this should be about 0.27
print(cm_both.loc['Generalized Anxiety Disorder', 'Generalized Anxiety Disorder'])

# Table 1: the four most common predictions for each class
rows = []
for cls in anxiety_four:
    top4 = cm_both.loc[cls].sort_values(ascending=False).head(4)
    row = {'true_class': cls}
    for i in range(4):
        row[f'pred_{i+1}'] = f"{top4.index[i]} ({top4.iloc[i]:.2f})"
    rows.append(row)
pd.DataFrame(rows)

0.2655367231638418


,true_class,pred_1,pred_2,pred_3,pred_4
0,Anxiety,Anxiety (0.78),Depression (0.13),Panic Disorder (0.03),Bipolar Disorder (0.02)
1,Anxiety and Stress,Anxiety and Stress (0.49),Depression (0.39),Anxiety (0.05),Panic Disorder (0.03)
2,Generalized Anxiety Disorder,Anxiety (0.34),Generalized Anxiety Disorder (0.27),Depression (0.24),Anxiety and Stress (0.06)
3,Panic Disorder,Panic Disorder (0.39),Anxiety (0.38),Depression (0.14),Anxiety and Stress (0.09)


In [3]:
# Table 2: share correct, share staying inside the anxiety family, share going to Depression
summary = pd.DataFrame({
    'correct': [cm_both.loc[c, c] for c in anxiety_four],
    'to_other_anxiety_classes': [
        cm_both.loc[c, [x for x in anxiety_four if x != c]].sum() for c in anxiety_four
    ],
    'to_Depression': [cm_both.loc[c, 'Depression'] for c in anxiety_four],
}, index=anxiety_four)
summary.round(2)

,correct,to_other_anxiety_classes,to_Depression
Anxiety,0.78,0.04,0.13
Anxiety and Stress,0.49,0.09,0.39
Generalized Anxiety Disorder,0.27,0.41,0.24
Panic Disorder,0.39,0.46,0.14


In [4]:
short_df = pd.read_csv('data/short_review_rows.csv')   # nothing is written
print(short_df.shape)                                  # expect 623 rows
print(short_df.columns.tolist())

(623, 9)
['Unnamed: 0', 'uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_clean']


In [5]:
TEXT_COL = 'review_clean'   # change if the table only has 'review' (apply the same cleaning first)

# 28-class labels, using the same merge_map as before
short_df['condition_merged'] = short_df['condition'].replace(merge_map)
y_short = short_df['condition_merged']

# Check 1: labels that are NOT among the 28 training classes (expect an empty set)
print('Labels not in training classes:', set(y_short) - set(y_train))

# Check 2: rows whose drug never appears in train
print('Rows with an unseen drug:', (~short_df[DRUG_COL].isin(train_df[DRUG_COL])).sum())

# Inputs: the already-fitted vectorizer and encoder, TRANSFORM only
X_short = vectorizer.transform(short_df[TEXT_COL])
drug_short = encoder.transform(short_df[[DRUG_COL]])
X_short_both = hstack([X_short, drug_short]).tocsr()

# Accuracy for the three models, plus the "always Birth Control" reference
results = pd.DataFrame({
    'model': ['text-only', 'drug-only', 'text + drug', 'always Birth Control'],
    'accuracy': [
        accuracy_score(y_short, model.predict(X_short)),
        accuracy_score(y_short, model_drug.predict(drug_short)),
        accuracy_score(y_short, model_both.predict(X_short_both)),
        (y_short == 'Birth Control').mean(),
    ]
})
print(results.round(3))

# Class counts (to see how many classes have only a handful of rows)
y_short.value_counts()

Labels not in training classes: set()
Rows with an unseen drug: 7
                  model  accuracy
0             text-only     0.281
1             drug-only     0.785
2           text + drug     0.809
3  always Birth Control     0.034


condition_merged
Pain                            129
Depression                       76
Anxiety                          70
Insomnia                         36
ADHD                             28
Bipolar Disorder                 28
High Blood Pressure              22
Birth Control                    21
Weight Management                20
Erectile Dysfunction             19
Panic Disorder                   19
Muscle Spasm                     19
Diabetes, Type 2                 19
Anxiety and Stress               15
Osteoarthritis                   14
Acne                             12
Urinary Tract Infection          10
Fibromyalgia                     10
Migraine                          9
Generalized Anxiety Disorder      8
Constipation                      7
Smoking Cessation                 7
Emergency Contraception           7
Vaginal Yeast Infection           5
Bacterial Infection               4
Migraine Prevention               4
Abnormal Uterine Bleeding         3
Bowel Prepa

In [6]:
# Optional: Generalized Anxiety Disorder reviews that the final model calls Depression
gad_to_dep = val_df[(y_val == 'Generalized Anxiety Disorder') & (y_pred_both == 'Depression')]
print('Reviews:', len(gad_to_dep))                         # roughly 42

# 1. Which drugs do they mention?
print(gad_to_dep[DRUG_COL].value_counts().head(8))

# 2. How are those drugs labelled in train overall? (a few condition columns only)
top_drugs = gad_to_dep[DRUG_COL].value_counts().head(5).index
cols = ['Depression', 'Anxiety', 'Generalized Anxiety Disorder', 'Panic Disorder', 'Anxiety and Stress']
pd.crosstab(train_df.loc[train_df[DRUG_COL].isin(top_drugs), DRUG_COL],
            train_df['condition_merged'])[cols]

Reviews: 43
drugName
Venlafaxine     9
Escitalopram    9
Lexapro         5
Effexor XR      4
Sertraline      4
Cymbalta        3
Duloxetine      3
Paroxetine      3
Name: count, dtype: int64


condition_merged,Depression,Anxiety,Generalized Anxiety Disorder,Panic Disorder,Anxiety and Stress
drugName,,,,,
Effexor XR,82,51,22,7,8
Escitalopram,240,263,179,0,0
Lexapro,157,185,93,0,0
Sertraline,317,0,49,121,163
Venlafaxine,287,79,46,17,34


In [7]:
# Optional: Generalized Anxiety Disorder reviews that the final model calls Depression
gad_to_dep = val_df[(y_val == 'Anxiety and Stress') & (y_pred_both == 'Depression')]
print('Reviews:', len(gad_to_dep))                         # roughly 42

# 1. Which drugs do they mention?
print(gad_to_dep[DRUG_COL].value_counts().head(8))

# 2. How are those drugs labelled in train overall? (a few condition columns only)
top_drugs = gad_to_dep[DRUG_COL].value_counts().head(5).index
cols = ['Depression', 'Anxiety', 'Generalized Anxiety Disorder', 'Panic Disorder', 'Anxiety and Stress']
pd.crosstab(train_df.loc[train_df[DRUG_COL].isin(top_drugs), DRUG_COL],
            train_df['condition_merged'])[cols]

Reviews: 95
drugName
Fluoxetine     18
Citalopram     16
Venlafaxine    10
Celexa         10
Sertraline     10
Prozac          8
Effexor         4
Luvox           4
Name: count, dtype: int64


condition_merged,Depression,Anxiety,Generalized Anxiety Disorder,Panic Disorder,Anxiety and Stress
drugName,,,,,
Celexa,101,0,0,3,104
Citalopram,234,0,65,34,272
Fluoxetine,191,0,0,26,140
Sertraline,317,0,49,121,163
Venlafaxine,287,79,46,17,34


In [ ]:
train_df.head()

In [9]:
import sys, sklearn, numpy, scipy, joblib   # the libraries the saved files depend on

print(sys.version)                           # your Python version
print("scikit-learn", sklearn.__version__)
print("numpy", numpy.__version__)
print("scipy", scipy.__version__)
print("joblib", joblib.__version__)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
scikit-learn 1.7.2
numpy 2.3.5
scipy 1.16.3
joblib 1.5.2


In [10]:
   print(len(model.classes_), model.n_features_in_)            # expect 28 and 34186 (text only)
   print(len(model_both.classes_), model_both.n_features_in_)  # expect 28 and 35346 (text + drug)

28 34186
28 35346
